In [1]:
# Installing all packages needed
import subprocess
import sys

packages = [
    "pandas==2.2.0",
    "numpy==1.26.4",
    "scikit-learn==1.4.0",
    "shap==0.44.1",
    "langchain==0.1.9",
    "langchain-community==0.0.24",
    "langchain-groq==0.1.3",
    "groq==0.4.2",
    "joblib==1.3.2",
    "python-dotenv==1.0.1"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")

All packages installed.


In [3]:
# Imports all libraries into memory and loads your Groq API key from the .env file.
import pandas as pd
import numpy as np
import joblib
import shap
import os
import json
import warnings

warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from groq   import Groq

load_dotenv("../.env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(f"pandas       : {pd.__version__}")
print(f"numpy        : {np.__version__}")
print(f"shap         : {shap.__version__}")
print(f"groq         : {__import__('groq').__version__}")
print(f"API Key      : {'Loaded' if GROQ_API_KEY else '❌ Not found - check .env file'}")

pandas       : 2.2.0
numpy        : 1.26.4
shap         : 0.44.1
groq         : 0.4.2
API Key      : Loaded


In [4]:
# Loads the trained Random Forest model and scaler saved in 3_feature_eng.
# Also loads the test data that the model has never seen before.
best_model = joblib.load("../models/best_model.pkl")
scaler     = joblib.load("../models/scaler.pkl")

X_train    = pd.read_csv("../data/X_train.csv")
X_test     = pd.read_csv("../data/X_test.csv")
y_test     = pd.read_csv("../data/y_test.csv").squeeze()

print(f"Model type      : {type(best_model).__name__}")
print(f"X_train shape   : {X_train.shape}")
print(f"X_test shape    : {X_test.shape}")
print(f"y_test shape    : {y_test.shape}")
print(f"Features        : {X_train.shape[1]}")
print(f"\nAll feature names:")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:>2}. {col}")

Model type      : RandomForestClassifier
X_train shape   : (20987, 29)
X_test shape    : (4495, 29)
y_test shape    : (4495,)
Features        : 29

All feature names:
   1. LIMIT_BAL
   2. SEX
   3. EDUCATION
   4. MARRIAGE
   5. AGE
   6. PAY_0
   7. PAY_2
   8. PAY_3
   9. PAY_4
  10. PAY_5
  11. PAY_6
  12. BILL_AMT1
  13. BILL_AMT2
  14. BILL_AMT3
  15. BILL_AMT4
  16. BILL_AMT5
  17. BILL_AMT6
  18. PAY_AMT1
  19. PAY_AMT2
  20. PAY_AMT3
  21. PAY_AMT4
  22. PAY_AMT5
  23. PAY_AMT6
  24. credit_utilization_rate
  25. debt_to_income_ratio
  26. avg_payment_delay
  27. total_bill_amt
  28. total_pay_amt
  29. payment_to_bill_ratio


In [ ]:
# Takes one sample customer from the test set and runs a prediction to confirm the model loads and predicts correctly before connecting anything else.
sample          = X_test.iloc[[0]]
prediction      = best_model.predict(sample)[0]
probability     = best_model.predict_proba(sample)[0]
actual          = y_test.iloc[0]

print(f"{'Feature':<30} {'Scaled Value':>14}")
print("-" * 47)
for col in sample.columns:
    print(f"  {col:<28} {sample[col].values[0]:>14.4f}")

print(f"\nPrediction Result:")
print(f"Predicted : {'DEFAULT' if prediction == 1 else 'NO DEFAULT'}")
print(f"No Default % : {probability[0]*100:.1f}%")
print(f"Default % : {probability[1]*100:.1f}%")
print(f"Actual Label : {'DEFAULT' if actual == 1 else 'NO DEFAULT'}")
print(f"Correct : {'Yes' if prediction == actual else 'No'}")

Feature                          Scaled Value
-----------------------------------------------
  LIMIT_BAL                           -1.0592
  SEX                                  1.0000
  EDUCATION                            0.0000
  MARRIAGE                             0.0000
  AGE                                  0.8148
  PAY_0                               -1.0000
  PAY_2                               -1.0000
  PAY_3                               -1.0000
  PAY_4                               -2.0000
  PAY_5                               -2.0000
  PAY_6                               -2.0000
  BILL_AMT1                           -0.7220
  BILL_AMT2                           -0.7126
  BILL_AMT3                           -0.7173
  BILL_AMT4                           -0.7075
  BILL_AMT5                           -0.6992
  BILL_AMT6                           -0.6874
  PAY_AMT1                            -0.4508
  PAY_AMT2                            -0.4924
  PAY_AMT3                      

In [15]:
# Checking best models available from the groq (free tire)
import requests
import os
from dotenv import load_dotenv

load_dotenv("../.env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

url     = "https://api.groq.com/openai/v1/models"
headers = {
    "Authorization" : f"Bearer {GROQ_API_KEY}",
    "Content-Type"  : "application/json"
}

response = requests.get(url, headers=headers)
models   = response.json()

print("Available models on your Groq account:")
print("=" * 55)
for model in models["data"]:
    print(f"  → {model['id']}")

Available models on your Groq account:
  → meta-llama/llama-prompt-guard-2-86m
  → openai/gpt-oss-safeguard-20b
  → meta-llama/llama-prompt-guard-2-22m
  → qwen/qwen3.8-27b
  → allam-2-7b
  → canopylabs/orpheus-v1-english
  → openai/gpt-oss-20b
  → whisper-large-v3-turbo
  → groq/compound
  → openai/gpt-oss-120b
  → canopylabs/orpheus-arabic-saudi
  → qwen/qwen3.6-27b
  → whisper-large-v3
  → groq/compound-mini


In [18]:
# Connects to Groq API using your API key and sends a test message to confirm the connection works before using it in the full agent.
# It automatically picks the best available model from your account
import requests
import os
from groq    import Groq
from dotenv  import load_dotenv

load_dotenv("../.env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Step 1: Fetch all available models automatically
url     = "https://api.groq.com/openai/v1/models"
headers = {
    "Authorization" : f"Bearer {GROQ_API_KEY}",
    "Content-Type"  : "application/json"
}

response     = requests.get(url, headers=headers)
models_data  = response.json()
model_ids    = [m["id"] for m in models_data["data"]]

print("Models available on your account:")
for m in model_ids:
    print(f"  → {m}")

# Step 2: Pick best available model automatically
preferred_models = [
    "llama-3.3-70b-versatile",
    "llama-3.1-70b-versatile",
    "llama3-70b-8192",
    "llama3-8b-8192",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
    "gemma-7b-it"
]

selected_model = None
for preferred in preferred_models:
    if preferred in model_ids:
        selected_model = preferred
        break

if selected_model is None:
    selected_model = model_ids[0]

print(f"\nSelected model : {selected_model}")

# Step 3: Create Groq client with selected model
groq_client = Groq(api_key=GROQ_API_KEY)

def ask_llm(prompt_text):
    """
    Sends prompt to Groq and returns text response.
    Uses the best available model automatically.
    """
    response = groq_client.chat.completions.create(
        model    = selected_model,
        messages = [
            {
                "role"    : "system",
                "content" : (
                    "You are a senior credit risk analyst "
                    "at a bank with 15 years of experience."
                )
            },
            {
                "role"    : "user",
                "content" : prompt_text
            }
        ],
        temperature = 0.1,
        max_tokens  = 1024
    )
    return response.choices[0].message.content

# Step 4: Test connection
test_response = ask_llm("Say the word CONNECTED.")

print(f"\nGroq API Status : Working")
print(f"Model used : {selected_model}")
print(f"Test response : {test_response}")

Models available on your account:
  → groq/compound
  → allam-2-7b
  → qwen/qwen3.6-27b
  → whisper-large-v3-turbo
  → canopylabs/orpheus-v1-english
  → openai/gpt-oss-120b
  → meta-llama/llama-prompt-guard-2-86m
  → qwen/qwen3.8-27b
  → whisper-large-v3
  → groq/compound-mini
  → openai/gpt-oss-20b
  → canopylabs/orpheus-arabic-saudi
  → openai/gpt-oss-safeguard-20b
  → meta-llama/llama-prompt-guard-2-22m

Selected model : groq/compound

Groq API Status : Working
Model used : groq/compound
Test response : **Reasoning**

The original request was simply to say the word **CONNECTED**.  
There is no additional context or requirement beyond reproducing that word.  
Therefore, the appropriate response is to provide the word itself, preceded by a brief explanation that this fulfills the instruction.

**Answer**

To follow the instructions, I simply need to say the word **CONNECTED**.

**CONNECTED**


In [19]:
#  Calculating SHAP values for all test customers.
# SHAP (SHapley Additive exPlanations) is a framework used in machine learning to explain the outputs of complex models (often called "black-box" models). 
# It quantifies how much each individual feature contributes to a specific prediction.
# SHAP tells us exactly how much each feature pushed the prediction toward or away from default for every single customer.
print("Calculating SHAP values for all test customers...")

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_for_default = shap_values[1]
else:
    shap_for_default = shap_values

print(f"\nSHAP calculation complete.")
print(f"Customers explained : {shap_for_default.shape[0]:,}")
print(f"Features per customer: {shap_for_default.shape[1]}")

sample_shap = shap_for_default[0]
shap_dict   = dict(zip(X_test.columns, sample_shap))
sorted_shap = sorted(
    shap_dict.items(),
    key    = lambda x: abs(x[1]),
    reverse= True
)

print(f"\nTop 10 SHAP factors for Customer 0:")
print(f"{'Rank':<6} {'Feature':<30} {'SHAP Score':>12}  {'Effect'}")
print("-" * 68)
for rank, (feature, value) in enumerate(sorted_shap[:10], 1):
    effect = " Pushes toward Default" if value > 0 \
            else " Pushes away from Default"
    print(f"  {rank:<4} {feature:<30} {value:>+12.4f}  {effect}")

Calculating SHAP values for all test customers...

SHAP calculation complete.
Customers explained : 4,495
Features per customer: 29

Top 10 SHAP factors for Customer 0:
Rank   Feature                          SHAP Score  Effect
--------------------------------------------------------------------
  1    PAY_0                               -0.0376   Pushes away from Default
  2    total_pay_amt                       +0.0309   Pushes toward Default
  3    avg_payment_delay                   -0.0286   Pushes away from Default
  4    debt_to_income_ratio                -0.0252   Pushes away from Default
  5    total_bill_amt                      +0.0215   Pushes toward Default
  6    PAY_2                               -0.0183   Pushes away from Default
  7    PAY_AMT6                            -0.0144   Pushes away from Default
  8    LIMIT_BAL                           +0.0130   Pushes toward Default
  9    PAY_AMT2                            +0.0116   Pushes toward Default
  10   PAY_AM

In [ ]:
# Creating the prompt template function that formats prediction results and SHAP values into a structured question for the LLM to answer.
def build_prompt(prediction, confidence,
                shap_summary, customer_profile):
    """
    Builds a structured prompt for the LLM.
    Takes model output and SHAP values
    and formats them into a clear question.
    """
    return f"""
A machine learning model analyzed a bank customer
and made the following credit risk prediction:

PREDICTION : {prediction}
CONFIDENCE : {confidence}

Top factors that influenced this decision (SHAP Analysis):
{shap_summary}

Customer Key Profile Values:
{customer_profile}

Instructions:
1. Explain WHY this prediction was made in exactly 3 clear bullet points using simple language that a non-technical bank manager can understand.
2. Suggest exactly 2 specific actions this customer can take to improve their credit risk profile.

Use this exact format:

**Why this prediction was made:**
- [Explanation point 1]
- [Explanation point 2]
- [Explanation point 3]

**Actions the customer can take:**
1. [Specific action 1]
2. [Specific action 2]
"""

print("Prompt builder function : Created")
print("Function name           : build_prompt()")
print("Parameters              : prediction, confidence,")
print("                          shap_summary, customer_profile")

Prompt builder function : ✅ Created
Function name           : build_prompt()
Parameters              : prediction, confidence,
                          shap_summary, customer_profile


In [21]:
# Building the complete AI agent function. When called with a customer index it runs the full pipeline 
# prediction -> SHAP -> prompt -> LLM explanation and returns everything as a structured result.
def explain_prediction(customer_index, X_data, y_data=None):
    """
    Full AI Agent pipeline for one customer.

    Steps:
    1. Get model prediction and probability
    2. Calculate SHAP values for this customer
    3. Format top factors into readable text
    4. Build prompt with all context
    5. Send to Groq LLM
    6. Return complete result dictionary

    Args:
        customer_index : Row number in X_data
        X_data         : Feature dataframe (X_test)
        y_data         : Actual labels (optional)

    Returns:
        dict containing prediction, probabilities,
        SHAP factors and AI explanation
    """

    # Step 1: Get prediction
    customer_row = X_data.iloc[[customer_index]]
    prediction = best_model.predict(customer_row)[0]
    probability = best_model.predict_proba(customer_row)[0]
    prediction_label = " DEFAULT RISK " if prediction == 1 \
                    else " NO DEFAULT "
    confidence       = round(max(probability) * 100, 1)

    # Step 2: Get SHAP values for this customer
    customer_shap = shap_for_default[customer_index]
    shap_dict = dict(zip(X_data.columns, customer_shap))
    sorted_shap = sorted(
        shap_dict.items(),
        key     = lambda x: abs(x[1]),
        reverse = True
    )[:8]

    # Step 3: Format SHAP into readable text
    shap_summary = ""
    for feature, value in sorted_shap:
        direction = "increases" if value > 0 else "decreases"
        impact    = "HIGH"   if abs(value) > 0.15 else \
                    "MEDIUM" if abs(value) > 0.05 else "LOW"
        shap_summary += (
            f"\n  - {feature}: "
            f"{direction} default risk "
            f"(Impact: {impact}, SHAP: {value:+.4f})"
        )

    # Step 4: Format customer profile
    key_features     = [
        "LIMIT_BAL", "AGE", "PAY_0", "PAY_2",
        "BILL_AMT1", "PAY_AMT1",
        "credit_utilization_rate",
        "avg_payment_delay"
    ]
    customer_profile = ""
    for feat in key_features:
        if feat in customer_row.columns:
            val = customer_row[feat].values[0]
            customer_profile += f"\n  - {feat}: {val:.4f}"

    # Step 5: Build prompt and send to LLM
    prompt         = build_prompt(
        prediction       = prediction_label,
        confidence       = f"{confidence}%",
        shap_summary     = shap_summary,
        customer_profile = customer_profile
    )
    ai_explanation = ask_llm(prompt)

    # Step 6: Build result dictionary
    result = {
        "customer_index"   : customer_index,
        "prediction"       : prediction_label,
        "confidence"       : f"{confidence}%",
        "probability"      : {
            "no_default"   : f"{probability[0]*100:.1f}%",
            "default"      : f"{probability[1]*100:.1f}%"
        },
        "top_shap_factors" : sorted_shap,
        "ai_explanation"   : ai_explanation
    }

    # Add actual label if provided
    if y_data is not None:
        actual = "DEFAULT" if y_data.iloc[customer_index] == 1 \
                else "NO DEFAULT"
        result["actual_label"] = actual
        result["correct"]      = (
            (prediction == 1 and actual == "DEFAULT") or
            (prediction == 0 and actual == "NO DEFAULT")
        )

    return result

print("AI Agent function : Ready")
print("Function name : explain_prediction()")
print("Pipeline : Prediction → SHAP → Prompt → LLM")

AI Agent function : Ready
Function name : explain_prediction()
Pipeline : Prediction → SHAP → Prompt → LLM


In [24]:
# Runs the complete AI agent on one customer and prints the full output including the AI-generated explanation.
# This is the first real test of the entire pipeline.
print("Running AI Agent on Customer 0...")

result = explain_prediction(
    customer_index = 0,
    X_data         = X_test,
    y_data         = y_test
)

print(f"Customer Index   : {result['customer_index']}")
print(f"Prediction       : {result['prediction']}")
print(f"Confidence       : {result['confidence']}")
print(f"No Default Prob  : {result['probability']['no_default']}")
print(f"Default Prob     : {result['probability']['default']}")
print(f"Actual Label     : {result['actual_label']}")
print(f"Correct          : {'Yes' if result['correct'] else 'No'}")

print(f"\nTop 8 SHAP Factors:")
print(f"{'Rank':<6} {'Feature':<30} {'Direction':<4} {'Score':>10}")
for rank, (feature, value) in enumerate(result["top_shap_factors"], 1):
    arrow = "up" if value > 0 else "Down"
    print(f"  {rank:<4} {feature:<30} {arrow:<4} {value:>+10.4f}")

print(f"\nAI Generated Explanation:")

print(result["ai_explanation"])

Running AI Agent on Customer 0...
Customer Index   : 0
Prediction       :  NO DEFAULT 
Confidence       : 55.1%
No Default Prob  : 55.1%
Default Prob     : 44.9%
Actual Label     : NO DEFAULT
Correct          : Yes

Top 8 SHAP Factors:
Rank   Feature                        Direction      Score
  1    PAY_0                          Down    -0.0376
  2    total_pay_amt                  up      +0.0309
  3    avg_payment_delay              Down    -0.0286
  4    debt_to_income_ratio           Down    -0.0252
  5    total_bill_amt                 up      +0.0215
  6    PAY_2                          Down    -0.0183
  7    PAY_AMT6                       Down    -0.0144
  8    LIMIT_BAL                      up      +0.0130

AI Generated Explanation:
**Why this prediction was made:**
- The customer’s recent payment record (PAY_0 and PAY_2) is strong, showing on‑time payments, which pulls the default risk down.  
- Their debt load is modest relative to income and credit limits (low debt‑to‑inc

In [ ]:
# Runing the AI agent on 5 different customers and shows a summary table of all results.
# Tests both DEFAULT RISK and NO DEFAULT cases.
print("Running AI Agent on 5 customers...")

test_indices = [0, 5, 10, 25, 50]
all_results  = []

for idx in test_indices:
    result = explain_prediction(
        customer_index = idx,
        X_data         = X_test,
        y_data         = y_test
    )
    all_results.append(result)

    correct_flag = "True" if result["correct"] else "False"
    print(f"  Customer {idx:<5}"
        f" → {result['prediction']:<22}"
        f" Confidence: {result['confidence']:<8}"
        f" Correct: {correct_flag}")

correct_count = sum(1 for r in all_results if r["correct"])

print(f"\nResults Summary:")
print(f"  Customers tested    : {len(all_results)}")
print(f"  Correct predictions : {correct_count}/{len(all_results)}")
print(f"  Sample accuracy     : {correct_count/len(all_results)*100:.0f}%")

print(f"\nDetailed Probabilities:")
print(f"{'Customer':<12} {'No Default':>12} {'Default':>10} {'Actual':>12}")

for r in all_results:
    print(f"  {r['customer_index']:<10}"
        f" {r['probability']['no_default']:>12}"
        f" {r['probability']['default']:>10}"
        f" {r['actual_label']:>12}")

Running AI Agent on 5 customers...
  Customer 0     →  NO DEFAULT            Confidence: 55.1%    Correct: True
  Customer 5     →  DEFAULT RISK          Confidence: 92.6%    Correct: True
  Customer 10    →  NO DEFAULT            Confidence: 72.2%    Correct: True
  Customer 25    →  NO DEFAULT            Confidence: 78.7%    Correct: True
  Customer 50    →  NO DEFAULT            Confidence: 80.6%    Correct: True

Results Summary:
  Customers tested    : 5
  Correct predictions : 5/5
  Sample accuracy     : 100%

Detailed Probabilities:
Customer       No Default    Default       Actual
--------------------------------------------------
  0                 55.1%      44.9%   NO DEFAULT
  5                  7.4%      92.6%      DEFAULT
  10                72.2%      27.8%   NO DEFAULT
  25                78.7%      21.3%   NO DEFAULT
  50                80.6%      19.4%   NO DEFAULT


In [25]:
# Saveing all agent results into two files.
# A JSON file for structured data that can be read by other systems.
# A text file that is human readable for bank managers or team reviews.
os.makedirs("../reports", exist_ok=True)

output_records = []
for result in all_results:
    record = {
        "customer_index" : result["customer_index"],
        "prediction"     : result["prediction"],
        "confidence"     : result["confidence"],
        "probability"    : result["probability"],
        "actual_label"   : result["actual_label"],
        "correct"        : result["correct"],
        "top_factors"    : [
            {
                "feature"    : f,
                "shap_score" : round(v, 4),
                "direction"  : "toward_default" if v > 0 \
                            else "away_from_default"
            }
            for f, v in result["top_shap_factors"]
        ],
        "ai_explanation" : result["ai_explanation"]
    }
    output_records.append(record)

json_path = "../reports/agent_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(output_records, f, indent=2, ensure_ascii=False)

report_path = "../reports/agent_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("CREDIT RISK AI EXPLANATION REPORT\n")
    f.write("=" * 60 + "\n\n")

    for result in all_results:
        f.write(f"Customer Index : {result['customer_index']}\n")
        f.write(f"Prediction     : {result['prediction']}\n")
        f.write(f"Confidence     : {result['confidence']}\n")
        f.write(f"No Default %   : {result['probability']['no_default']}\n")
        f.write(f"Default %      : {result['probability']['default']}\n")
        f.write(f"Actual Label   : {result['actual_label']}\n")
        f.write(f"Correct        : {result['correct']}\n")
        f.write(f"\nTop SHAP Factors:\n")
        for feat, val in result["top_shap_factors"]:
            arrow = "↑" if val > 0 else "↓"
            f.write(f"  {arrow} {feat}: {val:+.4f}\n")
        f.write(f"\nAI Explanation:\n")
        f.write(result["ai_explanation"])
        f.write("\n\n" + "-" * 60 + "\n\n")

json_size   = os.path.getsize(json_path)
report_size = os.path.getsize(report_path)

print(f"Files saved successfully:")
print(f" reports/agent_results.json  → {json_size:,} bytes")
print(f" reports/agent_report.txt    → {report_size:,} bytes")

Files saved successfully:
 reports/agent_results.json  → 10,814 bytes
 reports/agent_report.txt    → 6,891 bytes


In [27]:
#  Final verification that all files exist, all systems worked, and 4_ai_agent is fully complete.
import os

print("4_ai_agent COMPLETION CHECK")

required_files = {
    "../models/best_model.pkl"      : "Best model (Random Forest)",
    "../models/scaler.pkl"          : "Fitted scaler",
    "../reports/agent_results.json" : "Agent JSON results",
    "../reports/agent_report.txt"   : "Agent text report"
}

all_good = True
for path, label in required_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f" Done {label:<30} {size:>12,} bytes")
    else:
        print(f" NO {label:<30} NOT FOUND")
        all_good = False

print(f"\nAgent Test Results:")
print(f"{'Customer':<12} {'Prediction':<25} {'Conf':>8} {'Actual':<14} {'OK'}")

for r in all_results:
    flag = "True" if r["correct"] else "False"
    print(f"  {r['customer_index']:<10} "
        f"{r['prediction']:<25} "
        f"{r['confidence']:>8} "
        f"{r['actual_label']:<14} "
        f"{flag}")

correct_total = sum(1 for r in all_results if r["correct"])

print(f"\nSystem Status:")
print(f"  Groq LLM        : Connected (llama-3.1-8b-instant)")
print(f"  SHAP Explainer  : Working   ({shap_for_default.shape[0]:,} customers)")
print(f"  AI Agent        : Working   ({correct_total}/{len(all_results)} correct)")
print(f"  4_ai_agent : {'COMPLETE' if all_good else ' FIX ITEMS ABOVE'}")

4_ai_agent COMPLETION CHECK
 Done Best model (Random Forest)       15,316,441 bytes
 Done Fitted scaler                         1,751 bytes
 Done Agent JSON results                   10,814 bytes
 Done Agent text report                     6,891 bytes

Agent Test Results:
Customer     Prediction                    Conf Actual         OK
  0           NO DEFAULT                  55.1% NO DEFAULT     True
  5           DEFAULT RISK                92.6% DEFAULT        True
  10          NO DEFAULT                  72.2% NO DEFAULT     True
  25          NO DEFAULT                  78.7% NO DEFAULT     True
  50          NO DEFAULT                  80.6% NO DEFAULT     True

System Status:
  Groq LLM        : Connected (llama-3.1-8b-instant)
  SHAP Explainer  : Working   (4,495 customers)
  AI Agent        : Working   (5/5 correct)
  4_ai_agent : COMPLETE
